# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access full metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate all available record sets in the dataset and list their `@id`, name, and description, as well as their fields and column `@id`s.

In [ ]:
# List all record sets and fields, referencing entities by their `@id`
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f'RecordSet @id: {rs.id}')
        print(f'  Name: {getattr(rs, "name", None)}')
        print(f'  Description: {getattr(rs, "description", None)}')
        print('  Fields:')
        for field in rs.fields:
            print(f'    Field @id: {field.id} | Name: {getattr(field, "name", None)} | DataType: {getattr(field, "data_type", None)}')
            if hasattr(field, "column"):
                print(f'      Column @id: {field.column.id}')
        print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract data from each record set by its `@id`, storing each as a DataFrame. You can modify `record_set_ids` to focus on specific record sets discovered above.

In [ ]:
# Collect record set IDs from dataset metadata
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print available columns for each DataFrame
for rs_id in record_set_ids:
    print(f'Columns for RecordSet {rs_id}:')
    print(dataframes[rs_id].columns.tolist())
    print(dataframes[rs_id].head(2))
    print('---')

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

Below is an example EDA for a numeric field in a selected record set. **Ensure to reference fields by their `@id` as shown in the overview.**

In [ ]:
# Example: Select a record set and numeric field for EDA

if record_set_ids:
    record_set_id = record_set_ids[0]  # Choose the first record set as an example
    df = dataframes[record_set_id]

    # Identify numeric fields from metadata
    numeric_field_id = None
    group_field_id = None
    for field in dataset.record_set(record_set_id).fields:
        if getattr(field, "data_type", "") in ["schema:Float", "schema:Integer", "schema:Number"]:
            numeric_field_id = field.id
            break
    for field in dataset.record_set(record_set_id).fields:
        if getattr(field, "data_type", "") == "schema:Text":
            group_field_id = field.id
            break

    # Apply filtering if numeric field exists
    if numeric_field_id and numeric_field_id in df.columns and not df[numeric_field_id].isnull().all():
        threshold = df[numeric_field_id].mean()  # Set threshold as mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by another field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example: plotting the distribution of a numeric field and the mean value grouped by a category, using the selected record set and fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for numeric field
if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped barplot if group_field available
    if group_field_id and group_field_id in df.columns:
        group_stats = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10,5))
        sns.barplot(data=group_stats, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key observations:**
- The dataset provides ordered logistic regression outputs and survey data for households in Samburu, Isiolo, and Marsabit counties, Northern Kenya.
- Record sets and fields are referenced using their `@id` for clarity and consistency.
- Exploratory analysis identified potential numeric features for further study, and initial filtering and normalization steps highlight household adoption predictors.
- Data visualizations reveal distributions and group differences that can inform policy analysis and research.

Further exploration can include correlational analyses, advanced modeling, or integration with additional datasets.